# Чекпоинт 7

Проект: «Идентификация искусственно сгенерированных текстов: разработка методов для обеспечения информационной достоверности».


## Краткая цель

Базовая цель — воспроизвести PAWN и проверить, дают ли дополнительные признаки от второй LLM прирост качества.

Основные направления доработок: вторая frozen LLM, cross-model метрики, метрики второй модели, fusion hidden states, sequence-level aggregate metrics.


## Импорты 

In [73]:
from pathlib import Path
import json
import polars as pl

pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(120)

ROOT = Path.cwd()

## 0.1 Датасет

В качестве датасета для всех экспериментов используется [MAGE](https://huggingface.co/datasets/yaful/MAGE/viewer/default/train?row=18):

- Paper: https://arxiv.org/pdf/2305.13242

- GitHub: https://github.com/yafuly/MAGE?tab=readme-ov-file#-dataset

- Hugging Face: https://huggingface.co/datasets/yaful/MAGE/viewer/default/train?row=18

Датасет содержит большое количество доменов и моделей, что позволяет полноценно оценить работу детектора.

Более того, этот датасет используется в оригинальной статье [PAWN](https://www.sciencedirect.com/science/article/pii/S156625352500538X?ref=pdf_download&fr=RR-2&rr=9f97f735c8398b88)

Изначально в датасете использовалось следующее распределение по категориям: 

1 - Human-written, 0 - Machine-generated

Чтобы соответствовать оригинальной работе, категории были поменяны местами: 

0 - Human-written, 1 - Machine-generated

In [74]:
df_mage_train = pl.read_csv("pawn++/dataset/MAGE/testbeds/cross_domains_cross_models/train.csv")
df_mage_valid = pl.read_csv("pawn++/dataset/MAGE/testbeds/cross_domains_cross_models/valid.csv")
df_mage_test = pl.read_csv("pawn++/dataset/MAGE/testbeds/cross_domains_cross_models/test.csv")

In [75]:
print(f"Кол-во наблюдений в трейне: {df_mage_train.height}")
print(f"Кол-во наблюдений в валидации: {df_mage_valid.height}")
print(f"Кол-во наблюдений в тесет: {df_mage_test.height}")

Кол-во наблюдений в трейне: 319071
Кол-во наблюдений в валидации: 56792
Кол-во наблюдений в тесет: 60743


In [76]:
# Соотношение классов в трейне
df_mage_train.group_by("label").agg(pl.len().alias("count"))

label,count
i64,u32
1,225753
0,93318


In [77]:
# Соотношение классов в валидации
df_mage_valid.group_by("label").agg(pl.len().alias("count"))

label,count
i64,u32
0,28799
1,27993


In [78]:
# Соотношение классов в тесте
df_mage_test.group_by("label").agg(pl.len().alias("count"))

label,count
i64,u32
1,30265
0,30478


### Важное замечание

Для проведения экспериментов использовалась сбалансированная подвыборка из трейна и валидации на 5000 и 2000 наблюдений, соответственно. Тестовая выборка использовалась полностью для замера финальных метрик.

In [79]:
df_mage_train_sampled = pl.read_csv("pawn++/dataset/MAGE/testbeds/cross_domains_cross_models/train_sampled_inv.csv")
df_mage_valid_sampled = pl.read_csv("pawn++/dataset/MAGE/testbeds/cross_domains_cross_models/valid_sampled_inv.csv")

In [80]:
# Соотношение классов в трейне
df_mage_train_sampled.group_by("label").agg(pl.len().alias("count"))

label,count
i64,u32
1,2500
0,2500


In [81]:
# Соотношение классов в валидации
df_mage_valid_sampled.group_by("label").agg(pl.len().alias("count"))

label,count
i64,u32
0,1000
1,1000


## 0.2 Модели

Для всех экспериментов использовалась модель ```meta-llama/Llama-3.2-1B-Instruct``` и ее базовая версия - ```meta-llama/Llama-3.2-1B```. 

## Загрузка результатов


In [82]:
RUNS = {
    "Llama Instruct baseline": {
        "group": "Baseline",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/runs/single_model/mage_llama_instuct/test_metrics.json",
    },
    "Llama Base baseline": {
        "group": "Baseline",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/runs/single_model/mage_llama_base/test_metrics.json",
    },
    "Llama Instruct full baseline": {
        "group": "Baseline",
        "split": "full",
        "path": "pawn++/experiments/MAGE/runs/single_model/mage_llama_instruct_full/test_metrics.json",
    },
    "Llama Base + agg metrics": {
        "group": "Aggregate metrics",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/runs/single_model/mage_llama_base_agg_metrics/test_metrics.json",
    },
    "Llama Instruct + agg metrics": {
        "group": "Aggregate metrics",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/runs/single_model/mage_llama_instruct_agg_metrics/test_metrics.json",
    },
    "Llama Base + uniform HS": {
        "group": "Hidden-state fusion",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/runs/single_model/mage_llama_base_hs_uniform/test_metrics.json",
    },
    "Llama Instruct + uniform HS": {
        "group": "Hidden-state fusion",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/runs/single_model/mage_llama_instruct_hs_uniform/test_metrics.json",
    },
    "Llama Instruct + agg + uniform HS": {
        "group": "Hidden-state fusion",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/runs/single_model/mage_llama_instruct_agg_metrics_hs_uniform/test_metrics.json",
    },
    "Two models + XPPL": {
        "group": "Two models",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/runs/two_models/mage_llama_instruct_llama_base_xppl/test_metrics.json",
    },
    "Two models + second metrics + XPPL": {
        "group": "Two models",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/runs/two_models/mage_llama_instruct_llama_base_metrics_xppl/test_metrics.json",
    },
    "Two models + second HS": {
        "group": "Two models",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/runs/two_models/mage_llama_instruct_llama_base_hs/test_metrics.json",
    },
    "Two models + metrics + XPPL + HS": {
        "group": "Two models",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/runs/two_models/mage_llama_instruct_llama_base_metrics_xppl_hs/test_metrics.json",
    },
    "Two models + metrics + XPPL + uniform HS": {
        "group": "Two models",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/runs/two_models/mage_llama_instruct_llama_base_metrics_xppl_hs_uniform/test_metrics.json",
    },
    "PAWN++ sampled": {
        "group": "PAWN++",
        "split": "sampled",
        "path": "pawn++/experiments/MAGE/runs/two_models/mage_llama_instruct_llama_base_metrics_xppl_hs_uniform_agg_metrics/test_metrics.json",
    },
    "PAWN++ full": {
        "group": "PAWN++",
        "split": "full",
        "path": "pawn++/experiments/MAGE/runs/two_models/mage_llama_instruct_llama_base_metrics_xppl_hs_uniform_agg_metrics_full/test_metrics.json",
    },
}


In [83]:
def load_results(experiments: list[str]) -> pl.DataFrame:
    rows = []
    for experiment in experiments:
        run = RUNS[experiment]
        metrics = json.loads((ROOT / run["path"]).read_text())
        rows.append({
            "group": run["group"],
            "experiment": experiment,
            "split": run["split"],
            "AUCROC": metrics["test_roc_auc"],
            "accuracy": metrics["test_accuracy"],
            "human_recall": metrics["test_human_recall"],
            "ai_recall": metrics["test_ai_recall"],
            "f1_macro": metrics["test_f1_macro"],
        })

    return (
        pl.DataFrame(rows)
        .with_columns(pl.col("AUCROC", "accuracy", "human_recall", "ai_recall", "f1_macro").round(4))
    )


## Оригинальная PAWN-архитектура

Оригинальный PAWN берет logits и hidden states одной замороженной LLM. Из logits считаются token-level метрики, hidden states используются gate-сетью для взвешивания токенов, после чего агрегированный вектор передается в Aggregate NN.


![Original PAWN](pawn++/pawn_images/pawn_original.jpg)


### Бейзлайны

Базовая модель PAWN без доработок на подвыборке из 5000 тысяч наблюдений и на полном датасете MAGE (full)


In [84]:
load_results([
    "Llama Instruct baseline",
    "Llama Base baseline",
    "Llama Instruct full baseline",
])


group,experiment,split,AUCROC,accuracy,human_recall,ai_recall,f1_macro
str,str,str,f64,f64,f64,f64,f64
"""Baseline""","""Llama Instruct baseline""","""sampled""",0.8753,0.8003,0.7435,0.8568,0.7996
"""Baseline""","""Llama Base baseline""","""sampled""",0.8953,0.8041,0.7765,0.8316,0.8039
"""Baseline""","""Llama Instruct full baseline""","""full""",0.9758,0.9285,0.9494,0.9077,0.9285


## 1. Вторая frozen LLM

Основная идея всего расширения оригинального PAWN — добавить вторую замороженную модель. Она может отдавать свои logits и hidden states, на основе которых можно считать дополнительные признаки для улучшения ихсодной архитектуры PAWN.

![Second Frozen Model](pawn++/pawn_images/1_second_frozen_model.png)


## 2. Cross-model metrics (XPPL)

Был добавлен признак XPPL в стиле [Binoculars](https://arxiv.org/abs/2401.12070): одна модель дает log-probabilities, другая — probability distribution.


![Cross Metrics XPPL](pawn++/pawn_images/2_cross_metrics_xppl.png)


### Результаты XPPL


In [85]:
load_results([
    "Llama Instruct baseline",
    "Llama Base baseline",
    "Two models + XPPL",
    "Two models + second metrics + XPPL",
])


group,experiment,split,AUCROC,accuracy,human_recall,ai_recall,f1_macro
str,str,str,f64,f64,f64,f64,f64
"""Baseline""","""Llama Instruct baseline""","""sampled""",0.8753,0.8003,0.7435,0.8568,0.7996
"""Baseline""","""Llama Base baseline""","""sampled""",0.8953,0.8041,0.7765,0.8316,0.8039
"""Two models""","""Two models + XPPL""","""sampled""",0.8676,0.7848,0.7235,0.8457,0.7839
"""Two models""","""Two models + second metrics + XPPL""","""sampled""",0.9092,0.8259,0.78,0.8713,0.8254


Комментарий: XPPL отдельно оказался слабым, но результат заметно улучшился при добавлении token-level метрик второй модели. Это значит, что один cross-score недостаточно информативен, но в связке с полным набором признаков второй модели он становится полезнее.


## 3. Метрики второй модели

В базовой реализации используются метрики, рассчитанные по logits основной модели: `max_log_probs`, `entropy`, `next_token_log_probs`, `rank`, `top_p`.

Для второй frozen LLM были добавлены собственные token-level метрики. Они конкатенируются с метриками основной модели и проходят через Metrics NN.


![Second Model Metrics](pawn++/pawn_images/3_second_model_metrics.png)


### Результаты second-model metrics


In [86]:
load_results([
    "Llama Instruct baseline",
    "Llama Base baseline",
    "Two models + XPPL",
    "Two models + second metrics + XPPL",
    "Two models + metrics + XPPL + HS",
])


group,experiment,split,AUCROC,accuracy,human_recall,ai_recall,f1_macro
str,str,str,f64,f64,f64,f64,f64
"""Baseline""","""Llama Instruct baseline""","""sampled""",0.8753,0.8003,0.7435,0.8568,0.7996
"""Baseline""","""Llama Base baseline""","""sampled""",0.8953,0.8041,0.7765,0.8316,0.8039
"""Two models""","""Two models + XPPL""","""sampled""",0.8676,0.7848,0.7235,0.8457,0.7839
"""Two models""","""Two models + second metrics + XPPL""","""sampled""",0.9092,0.8259,0.78,0.8713,0.8254
"""Two models""","""Two models + metrics + XPPL + HS""","""sampled""",0.9045,0.8318,0.8348,0.8288,0.8318


Комментарий: это одно из наиболее полезных расширений. Метрики второй модели добавляют информацию о том, как другая LLM оценивает те же токены, и это дает прирост относительно использования только XPPL.


## 4. Hidden states fusion

В оригинальном PAWN используется последний hidden state. 

В улучшенной версии были проверены варианты использования информации сразу из всех hidden-state. Это реализовано за счет усреднения предварительно отнормализованных hidden states модели по каждому токену. Это делается параллельно для двух моделей, после чего их hidden states конкатенируются и подаются в Weights NN вместе в вектором, кодирующим позицию.


Для модели $m$ и токена $t$ усредненный hidden-state считается так:

$$
\tilde{h}^{(m)}_t =
\frac{1}{K}
\sum_{k=1}^{K}
\operatorname{LayerNorm}\left(h^{(m,k)}_t\right),
$$

где $h^{(m,k)}_t$ — hidden state токена $t$ на слое $k$, $K$ — число используемых слоев модели, а $\tilde{h}^{(m)}_t$ — итоговый усредненый hidden state.

Для двух моделей вход в Weights NN формируется как:

$$
g_t =
\left[
\tilde{h}^{(1)}_t ;
\tilde{h}^{(1)}_{t+1} ;
\tilde{h}^{(2)}_t ; 
\tilde{h}^{(2)}_{t+1} ;
p_t
\right]
$$

где $p_t$ — вектор с позиционным кодированием токена.

![Hidden States Fusion](pawn++/pawn_images/4_hidden_states_fusion.png)


### Результаты hidden-state fusion


In [87]:
load_results([
    "Llama Instruct baseline",
    "Llama Base baseline",
    "Llama Base + uniform HS",
    "Llama Instruct + uniform HS",
    "Two models + metrics + XPPL + uniform HS",
])


group,experiment,split,AUCROC,accuracy,human_recall,ai_recall,f1_macro
str,str,str,f64,f64,f64,f64,f64
"""Baseline""","""Llama Instruct baseline""","""sampled""",0.8753,0.8003,0.7435,0.8568,0.7996
"""Baseline""","""Llama Base baseline""","""sampled""",0.8953,0.8041,0.7765,0.8316,0.8039
"""Hidden-state fusion""","""Llama Base + uniform HS""","""sampled""",0.9273,0.8469,0.8368,0.857,0.8469
"""Hidden-state fusion""","""Llama Instruct + uniform HS""","""sampled""",0.9096,0.8244,0.794,0.8545,0.8242
"""Two models""","""Two models + metrics + XPPL + uniform HS""","""sampled""",0.9288,0.8557,0.8503,0.8609,0.8556


Комментарий: uniform fusion оказался самым стабильным улучшением. Особенно заметен результат `Llama Base + uniform HS`, который стал лучшим single-model вариантом на сэмплированной выборке.


## 5. Aggregated sequence-level metrics

Были добавлены sequence-level признаки: статистики по log-likelihood / surprisal всей последовательности и их производным. 

Исходя из работы [DivEye](https://arxiv.org/pdf/2509.18880), добавление агрегированных метрик на основе log-likelihood токенов может значительно улучшать качество детекторов.

Проверялись разные способы добавления агрегированных метрик в конец модели PAWN: 
- Конкатенация агрегированных метрик с финальным скором PAWN 
- Конкатенация агрегированных метрик с финальным вектором признаков перед подачей в `aggregate_nn`
- [FiLM](https://arxiv.org/pdf/1709.07871)-like добавление агрегированных метрик к финальному вектору признаков перед подачей в `aggregate_nn`

По итогам экспериментов лучший результат показал последний вариант. 

Математически пусть $F$ — финальный вектор признаков PAWN перед подачей в Aggregate NN.

Агрегированные sequence-level метрики обозначим как $s$. Сначала они нормализуются:

$$
\hat{s} = \operatorname{LayerNorm}(s).
$$

Затем из них предсказываются FiLM-параметры $\gamma$ и $\beta$:

$$
[\gamma, \beta] = W_{\text{film}} \hat{s} + b_{\text{film}},
$$

где $\gamma$ и $\beta$ имеют ту же размерность, что и $F$.

FiLM-like преобразование вектора $F$ считается как:

$$
F_{\text{film}} =
(1 + \gamma) \odot F + \beta.
$$

После этого уже модифицированный вектор передается в Aggregate NN:

$$
\text{logit} =
\operatorname{AggregateNN}(F_{\text{film}}).
$$

$W_{\text{film}}$ и  $b_{\text{film}}$ инициализируются нулями:

$$
W_{\text{film}} = 0, \quad b_{\text{film}} = 0.
$$

Тогда в начале обучения:

$$
\gamma = 0, \quad \beta = 0,
$$

и поэтому:

$$
F_{\text{film}} = F
$$

![Aggregated Metrics](pawn++/pawn_images/5_aggregated_metrics.png)


![Aggregate Metrics Fusion](pawn++/pawn_images/6_aggregated_metrics_fusion_film.png)

### Результаты


In [88]:
load_results([
    "Llama Instruct baseline",
    "Llama Base baseline",
    "Llama Instruct + agg metrics",
    "Llama Base + agg metrics",
])


group,experiment,split,AUCROC,accuracy,human_recall,ai_recall,f1_macro
str,str,str,f64,f64,f64,f64,f64
"""Baseline""","""Llama Instruct baseline""","""sampled""",0.8753,0.8003,0.7435,0.8568,0.7996
"""Baseline""","""Llama Base baseline""","""sampled""",0.8953,0.8041,0.7765,0.8316,0.8039
"""Aggregate metrics""","""Llama Instruct + agg metrics""","""sampled""",0.8821,0.8055,0.7766,0.8341,0.8053
"""Aggregate metrics""","""Llama Base + agg metrics""","""sampled""",0.8738,0.7862,0.7609,0.8113,0.786


Комментарий: агрегированные метрики не дали стабильного прироста в single-model setup. Качество для instruct-tuned модели выросло, однако для базовой модели, наоборот, упало. Вероятно, часть информации уже извлекается PAWN через token-level признаки, поэтому sequence-level статистики не дают значимого прироста по качеству.


## Финальная архитектура PAWN++

PAWN++ объединяет основные расширения: вторую frozen LLM, second-model token metrics, cross-model metrics / XPPL, hidden-state fusion и aggregate metrics.


![PAWN++](pawn++/pawn_images/pawn++.png)


### Итоговые результаты PAWN++


In [89]:
load_results([
    "Llama Instruct baseline",
    "Llama Base baseline",
    "PAWN++ sampled",
    "Llama Instruct full baseline",
    "PAWN++ full",
]).sort("AUCROC", descending=True)


group,experiment,split,AUCROC,accuracy,human_recall,ai_recall,f1_macro
str,str,str,f64,f64,f64,f64,f64
"""PAWN++""","""PAWN++ full""","""full""",0.9836,0.9515,0.972,0.9311,0.9515
"""Baseline""","""Llama Instruct full baseline""","""full""",0.9758,0.9285,0.9494,0.9077,0.9285
"""PAWN++""","""PAWN++ sampled""","""sampled""",0.9309,0.848,0.805,0.8908,0.8477
"""Baseline""","""Llama Base baseline""","""sampled""",0.8953,0.8041,0.7765,0.8316,0.8039
"""Baseline""","""Llama Instruct baseline""","""sampled""",0.8753,0.8003,0.7435,0.8568,0.7996


Комментарий: лучший результат получился у полной PAWN++ конфигурации на полном датасете. На сэмплированной выборке PAWN++ также оказался лучшим вариантом среди всех экспериментов, превосходя бейзлайны на более чем 3.5 процентных пункта.


## Финальная таблица

Все результаты собраны в одну таблицу и отсортированы по ROC-AUC.


In [90]:
results = load_results(list(RUNS.keys()))
results = results.sort("AUCROC", descending=True)
results

group,experiment,split,AUCROC,accuracy,human_recall,ai_recall,f1_macro
str,str,str,f64,f64,f64,f64,f64
"""PAWN++""","""PAWN++ full""","""full""",0.9836,0.9515,0.972,0.9311,0.9515
"""Baseline""","""Llama Instruct full baseline""","""full""",0.9758,0.9285,0.9494,0.9077,0.9285
"""PAWN++""","""PAWN++ sampled""","""sampled""",0.9309,0.848,0.805,0.8908,0.8477
"""Two models""","""Two models + metrics + XPPL + uniform HS""","""sampled""",0.9288,0.8557,0.8503,0.8609,0.8556
"""Hidden-state fusion""","""Llama Base + uniform HS""","""sampled""",0.9273,0.8469,0.8368,0.857,0.8469
"""Hidden-state fusion""","""Llama Instruct + uniform HS""","""sampled""",0.9096,0.8244,0.794,0.8545,0.8242
"""Two models""","""Two models + second metrics + XPPL""","""sampled""",0.9092,0.8259,0.78,0.8713,0.8254
"""Two models""","""Two models + metrics + XPPL + HS""","""sampled""",0.9045,0.8318,0.8348,0.8288,0.8318
"""Hidden-state fusion""","""Llama Instruct + agg + uniform HS""","""sampled""",0.9007,0.8173,0.7697,0.8646,0.8169


## Выводы

1. Базовый PAWN воспроизведен и используется как бейзлайн.
2. Самое стабильное single-model улучшение — усреднение hidden-states.
3. Эксперименты показали, что token-level метрики и cross-model признаки второй модели дают значительный прирост в качестве детекции.
4. Агрегированные метрики дают прирост не во всех экспериментах, однако в используются финальной архитектуре PAWN++ и позволяют получить более высокие метрики качества.
5. Модель PAWN++, обученная на полном MAGE датасете, превосходит базовую модель PAWN по всем ключевым метрикам.
